# Intelligent Video-Based Violence Detection

This notebook contains the Google Colab training workflow for the MobileNetV2 + Bidirectional LSTM model.

> The dataset and Kaggle credentials are not included in this repository. See the Training section in [`README.md`](../README.md) before running the notebook.
> The standalone local inference app is available in [`app.py`](../app.py).
> This project performs video-level classification from 16 uniformly sampled frames; it is not a real-time frame-by-frame detector.


# **تهيئة البيئة واستيراد المكتبات وتحديد الثوابت.**

In [ ]:
# 1. التثبيت والاستيراد (Setup & Imports)
!pip install tensorflow opencv-python scikit-learn gradio kaggle -q

import os
import math
import json
import random
import numpy as np
import cv2
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# إعدادات تسريع التدريب باستخدام Mixed Precision (اختياري حسب نوع GPU)
try:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print('✅ Using mixed precision policy: mixed_float16 (Faster Training)')
except Exception as e:
    print('⚠️ Mixed precision not available, using float32')

# الثوابت والتهيئة (Configuration)
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# أبعاد الصورة وطول التسلسل
IMAGE_HEIGHT, IMAGE_WIDTH = 128, 128
SEQUENCE_LENGTH = 16  # عدد الإطارات لكل فيديو
BATCH_SIZE = 8
EPOCHS = 20  # عدد دورات التدريب

# مسارات الملفات
DATA_DIR = './data/real-life-violence-situations-dataset'
WORK_DIR = './workdir'
os.makedirs(WORK_DIR, exist_ok=True)
MODEL_PATH = os.path.join(WORK_DIR, 'violence_model.h5')

print("✅ البيئة جاهزة وتم تحديد الثوابت.")

# **إعداد Kaggle API لتحميل البيانات.**

In [ ]:
# 2. إعداد Kaggle (Kaggle Setup)
# هذه الخطوة تطلب منك رفع ملف kaggle.json إذا لم يكن موجوداً
def setup_kaggle():
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json_path = kaggle_dir / 'kaggle.json'

    if not kaggle_json_path.exists():
        print("📂 يرجى رفع ملف kaggle.json الخاص بك الآن:")
        from google.colab import files
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            # نقل الملف للمسار الصحيح
            !mv kaggle.json /root/.kaggle/kaggle.json
            !chmod 600 /root/.kaggle/kaggle.json
            print("✅ تم إعداد Kaggle بنجاح.")
            return True
        else:
            print("❌ لم يتم رفع الملف.")
            return False
    return True

if setup_kaggle():
    # تحميل البيانات
    if not os.path.exists(DATA_DIR):
        print("⬇️ جاري تحميل البيانات من Kaggle...")
        !kaggle datasets download -d mohamedmustafa/real-life-violence-situations-dataset -p ./data --unzip
        print("✅ تم تحميل وفك ضغط البيانات.")
    else:
        print("✅ البيانات موجودة مسبقاً.")

# **دوال المعالجة المسبقة للفيديو (Preprocessing).**

In [ ]:
# 3. معالجة الفيديو (Video Processing Functions)

def extract_frames(video_path, seq_length=SEQUENCE_LENGTH):
    """
    تقوم هذه الدالة بقراءة الفيديو واستخراج عدد ثابت من الإطارات (16 إطار)،
    ثم تغيير حجمها ومعالجتها لتناسب شبكة MobileNetV2.
    """
    try:
        cap = cv2.VideoCapture(str(video_path))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # التأكد من صلاحية الفيديو
        if total_frames < seq_length:
            cap.release()
            return None

        # اختيار إطارات موزعة بانتظام على طول الفيديو
        indices = np.linspace(0, total_frames - 1, seq_length, dtype=int)
        frames = []

        for i in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if not ret:
                continue

            # تحويل الألوان من BGR إلى RGB
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            # تغيير الحجم
            frame = cv2.resize(frame, (IMAGE_WIDTH, IMAGE_HEIGHT))
            # المعالجة المسبقة الخاصة بـ MobileNet
            frame = preprocess_input(frame.astype('float32'))
            frames.append(frame)

        cap.release()

        if len(frames) < seq_length:
            return None

        return np.array(frames)
    except Exception as e:
        return None

# مولد البيانات (Data Generator) لتوفير الذاكرة
class VideoDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, paths, labels, batch_size, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.paths = paths
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.paths))
        self.skipped_paths = set()
        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.paths) / self.batch_size)

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        batch_x = []
        batch_y = []

        for i in batch_indices:
            video = extract_frames(self.paths[i])
            if video is not None:
                batch_x.append(video)
                batch_y.append(self.labels[i])
            else:
                self.skipped_paths.add(self.paths[i])

        if not batch_x:
            raise ValueError(f"No readable videos in batch {index}. Check the video files and preprocessing.")
        return np.asarray(batch_x, dtype=np.float32), np.asarray(batch_y, dtype=np.int64)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

print("✅ دوال المعالجة ومولد البيانات جاهزة.")

# **اكتشاف البيانات وتقسيمها.**

In [ ]:
# 4. Find the extracted dataset directory robustly
from pathlib import Path

DATA_ROOT = Path("./data")
expected_classes = {"NonViolence", "Violence"}
dataset_candidates = [
    folder for folder in DATA_ROOT.rglob("*")
    if folder.is_dir() and expected_classes.issubset({item.name for item in folder.iterdir() if item.is_dir()})
]

if not dataset_candidates:
    raise FileNotFoundError(
        "Could not find a folder containing both 'NonViolence' and 'Violence'. "
        "Check the dataset extraction described in DATASET.md."
    )

DATA_DIR = str(dataset_candidates[0])
print(f"Dataset directory: {DATA_DIR}")


In [ ]:
# 5. Build the video manifest and split it into train/validation sets

def load_data_paths(data_dir):
    paths, labels = [], []
    class_names = ["NonViolence", "Violence"]  # Keep this order aligned with app.py and the saved model.
    supported_extensions = {".mp4", ".avi", ".mov", ".mkv"}

    for label_idx, class_name in enumerate(class_names):
        class_dir = Path(data_dir) / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f"Missing class directory: {class_dir}")
        files = sorted(
            path for path in class_dir.iterdir()
            if path.is_file() and path.suffix.lower() in supported_extensions
        )
        paths.extend(str(path) for path in files)
        labels.extend([label_idx] * len(files))
        print(f"{class_name}: {len(files)} videos")

    if not paths:
        raise RuntimeError("No supported video files were found in the dataset folders.")
    return paths, labels, class_names

paths, labels, CLASS_NAMES = load_data_paths(DATA_DIR)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    paths, labels, test_size=0.2, stratify=labels, random_state=SEED
)

print(f"Training videos: {len(train_paths)}")
print(f"Validation videos: {len(val_paths)}")
print("Note: this is a random clip-level split, not a source-video/group split.")

train_gen = VideoDataGenerator(train_paths, train_labels, BATCH_SIZE)
val_gen = VideoDataGenerator(val_paths, val_labels, BATCH_SIZE, shuffle=False)


# **بناء هيكلية النموذج (CNN + LSTM).**

In [ ]:
# 6. بناء النموذج (Model Architecture)

def build_hybrid_model(num_classes):
    # الجزء الأول: استخراج الميزات المكانية (CNN)
    base_model = MobileNetV2(include_top=False, weights='imagenet', input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 3))

    # تجميد طبقات MobileNet لتسريع التدريب والحفاظ على الأوزان المدربة
    base_model.trainable = False

    inputs = layers.Input(shape=(SEQUENCE_LENGTH, IMAGE_HEIGHT, IMAGE_WIDTH, 3))

    # تطبيق CNN على كل إطار في الفيديو (TimeDistributed)
    x = layers.TimeDistributed(base_model)(inputs)
    x = layers.TimeDistributed(layers.GlobalAveragePooling2D())(x)

    # الجزء الثاني: تحليل الزمن (RNN/LSTM)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=False, dropout=0.3))(x)

    # طبقات التصنيف النهائية
    x = layers.BatchNormalization()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs, name="Violence_Detector_Net")
    return model

model = build_hybrid_model(len(CLASS_NAMES))
model.summary()

# تجميع النموذج
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# **بدء عملية التدريب.**

In [ ]:
# 7. تدريب النموذج (Training)

# إعدادات الحفظ والتوقف المبكر
callbacks_list = [
    callbacks.ModelCheckpoint(MODEL_PATH, save_best_only=True, monitor='val_accuracy', mode='max'),
    callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]

print("🚀 بدء التدريب...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks_list
)
print("✅ انتهى التدريب.")

## **الرسوم البيانية**

In [ ]:
# 8. الرسوم البيانية

import matplotlib.pyplot as plt

# التأكد من أن المتغير history موجود من الخلية السابقة
if 'history' in globals():
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc))

    plt.figure(figsize=(15, 5))

    # رسم الدقة (Accuracy)
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')

    # رسم الخسارة (Loss)
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')

    plt.show()
else:
    print("⚠️ لم يتم العثور على سجل التدريب history. تأكد من تشغيل خلية التدريب أولاً.")

# **التقييم ورسم مصفوفة الارتباك.**

In [ ]:
# 9. تقييم النموذج (Evaluation)

print("🔍 جاري حساب دقة النموذج على بيانات الاختبار...")
y_true = []
y_pred = []

# التكرار على بيانات التحقق للتنبؤ
for i in range(len(val_gen)):
    x_batch, y_batch = val_gen[i]
    preds = model.predict(x_batch, verbose=0)
    y_true.extend(y_batch)
    y_pred.extend(np.argmax(preds, axis=1))

print(f"\nSkipped validation videos (unreadable/too short): {len(val_gen.skipped_paths)}")

# عرض التقرير
print("\n📊 Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

print("\n🧩 Confusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

## **كود الحفظ**

In [ ]:
# 10. حفظ النموذج النهائي
model.save(MODEL_PATH)
print(f"✅ تم حفظ النموذج في المسار: {MODEL_PATH}")

# محاولة تنزيل النموذج تلقائياً إلى جهازك
try:
    from google.colab import files
    print("⬇️ جاري تحضير الملف للتنزيل...")
    files.download(MODEL_PATH)
except Exception as e:
    print(f"⚠️ لم يتم التنزيل تلقائياً (قد يكون بسبب إعدادات المتصفح). يمكنك تنزيله يدوياً من القائمة الجانبية: {MODEL_PATH}")

## **التحقق باستخدام عينات من البيانات**

In [ ]:
# 11. عرض عينات من التنبؤات (Sample Predictions)

# التأكد من تحميل النموذج أو توفره في الذاكرة
final_model = None
if os.path.exists(MODEL_PATH):
    try:
        final_model = tf.keras.models.load_model(MODEL_PATH)
        print("✅ تم تحميل النموذج من الملف.")
    except Exception as e:
        print(f"⚠️ تعذر تحميل النموذج من الملف: {e}")

if final_model is None and 'model' in globals():
    final_model = model
    print("⚠️ سيتم استخدام النموذج الموجود حالياً في الذاكرة.")

if final_model:
    print("🎥 عرض عينات من التنبؤات على بيانات التحقق:")

    # اختيار 5 عينات عشوائية (أو أقل إذا كانت البيانات قليلة)
    num_samples = min(5, len(val_paths))
    if num_samples > 0:
        sample_indices = np.random.choice(len(val_paths), num_samples, replace=False)

        for i, idx in enumerate(sample_indices):
            video_path = val_paths[idx]
            true_label_idx = val_labels[idx]
            true_label_name = CLASS_NAMES[true_label_idx]

            print(f"\n--- Sample {i+1} ---")
            print(f"مسار الفيديو: {video_path}")
            print(f"التصنيف الحقيقي: {true_label_name}")

            # استخراج الإطارات للفيديو
            frames = extract_frames(video_path)

            if frames is not None:
                # إضافة بعد batch size للتنبؤ بالنموذج
                input_data = np.expand_dims(frames, axis=0)

                # إجراء التنبؤ
                predictions = final_model.predict(input_data, verbose=0)[0]
                predicted_class_idx = np.argmax(predictions)
                predicted_class_name = CLASS_NAMES[predicted_class_idx]
                confidence = predictions[predicted_class_idx]

                print(f"التصنيف المتوقع: {predicted_class_name} (بثقة: {confidence:.2f})")

                # التحقق من صحة التنبؤ
                if predicted_class_idx == true_label_idx:
                    print("✅ تنبؤ صحيح")
                else:
                    print("❌ تنبؤ خاطئ")
            else:
                print("❌ لم نتمكن من معالجة الفيديو (عدد الإطارات غير كافٍ أو ملف تالف).")
    else:
        print("⚠️ لا توجد بيانات تحقق لعرض العينات.")
else:
    print("❌ النموذج غير متاح. يرجى تدريب النموذج أولاً.")

# **واجهة Gradio.**

## Run the inference app

The standalone Gradio application is maintained in [`app.py`](../app.py). After training or downloading the model into `models/violence_model.h5`, run `python app.py` from the repository root. The app binds to `127.0.0.1` and does not create a public Gradio share link by default.
